# 02 - Explore a session

**Audience:** you just ran a session and want to see what got captured.

Walks through quality, eye movements, pupil + LHIPA, per-phase report, and
post-hoc correction (if a profile JSON sits next to the CSV).

## What you'll get

- Metadata header (CoordinateOrigin, Profile, FileVersion).
- Data quality summary.
- Fixation/saccade detection with scanpath.
- Pupil + LHIPA.
- Per-phase report (skipped if not a SampleExperiment recording).
- Post-hoc correction preview (skipped if no profile JSON found).

## Getting your data into this notebook

`notebook_context()` auto-discovers a CSV in this order:

1. **Explicit path** — `notebook_context(csv="path/to/EyeTracking_*.csv")`
2. **Environment variable** — `export EYELEAN_CSV=path/to/file.csv`
3. **`Logs/` folder** — most-recent main `EyeTracking_*.csv` in any
   `Logs/` directory walking up from this notebook.
4. **Bundled sample** — falls back to a v1.2 SampleExperiment recording
   shipped in `Assets/StreamingAssets/` so the notebook runs end-to-end
   on a fresh checkout before you've recorded anything yourself.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Allow opening this notebook from a checkout without `pip install -e`.
_here = Path(os.getcwd()).resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "eyelean_analysis" / "__init__.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eyelean_analysis as ela

ctx = ela.notebook_context()
print(ctx)

## 1. Metadata + summary

In [ ]:
if ctx.metadata:
    print("CSV metadata header:")
    for k, v in ctx.metadata.items():
        print(f"  {k}: {v}")
data = ctx.data
summary = data.summary()
print()
for k, v in summary.items():
    if isinstance(v, list):
        v = f"[{len(v)} items] {v[:5]}{'...' if len(v) > 5 else ''}"
    print(f"  {k:<28} {v}")

## 2. Data quality

In [ ]:
df = data.df
ts = data.get_timestamps()
qm = ela.calculate_quality_metrics(
    timestamps=ts,
    validity=df["is_tracking_valid"].to_numpy() if "is_tracking_valid" in df.columns else None,
    left_validity=df["has_left_direction"].to_numpy() if "has_left_direction" in df.columns else None,
    right_validity=df["has_right_direction"].to_numpy() if "has_right_direction" in df.columns else None,
    left_pupil=df["left_pupil_diameter"].to_numpy() if "left_pupil_diameter" in df.columns else None,
    right_pupil=df["right_pupil_diameter"].to_numpy() if "right_pupil_diameter" in df.columns else None,
    left_openness=df["left_openness"].to_numpy() if "left_openness" in df.columns else None,
    right_openness=df["right_openness"].to_numpy() if "right_openness" in df.columns else None,
)
for k, v in qm.to_dict().items():
    if isinstance(v, float):
        print(f"  {k:<30} {v:.3f}")
    else:
        print(f"  {k:<30} {v}")

## 3. Eye-movement detection

Combined gaze direction is converted to angular yaw/pitch (degrees) before
feeding the I-VT classifier.

In [ ]:
needed = ("combined_dir_x", "combined_dir_y", "combined_dir_z")
movements = None
if all(c in df.columns for c in needed):
    dx = df[needed[0]].to_numpy(dtype=float)
    dy = df[needed[1]].to_numpy(dtype=float)
    dz = df[needed[2]].to_numpy(dtype=float)
    yaw = np.degrees(np.arctan2(dx, dz))
    pitch = -np.degrees(np.arcsin(np.clip(dy, -1, 1)))
    movements = ela.detect_eye_movements(yaw, pitch, ts)
    print(f"Fixations: {len(movements['fixations'])}")
    print(f"Saccades:  {len(movements['saccades'])}")
    if movements["fixations"]:
        durs_ms = [f.duration * 1000 for f in movements["fixations"]]
        print(f"  fixation dur ms - median {np.median(durs_ms):.1f}, p95 {np.percentile(durs_ms, 95):.1f}")
    if movements["saccades"]:
        amps = [s.amplitude for s in movements["saccades"]]
        print(f"  saccade amp deg - median {np.median(amps):.2f}, p95 {np.percentile(amps, 95):.2f}")
else:
    print("Missing combined-direction columns; skipping classifier.")

In [ ]:
if movements is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(yaw, pitch, color="lightgray", linewidth=0.4)
    if movements["fixations"]:
        fx = [f.centroid_x for f in movements["fixations"]]
        fy = [f.centroid_y for f in movements["fixations"]]
        sizes = [max(20, min(400, f.duration * 1000)) for f in movements["fixations"]]
        ax.scatter(fx, fy, s=sizes, alpha=0.6, edgecolor="black")
    ax.set_xlabel("yaw (deg)")
    ax.set_ylabel("pitch (deg)")
    ax.set_title(f"Scanpath - {ctx.csv_path.name}")
    ax.invert_yaxis()
    ax.set_aspect("equal", adjustable="datalim")
    plt.tight_layout()
    plt.show()

## 4. Pupil timeline + LHIPA

In [ ]:
if "left_pupil_diameter" in df.columns or "right_pupil_diameter" in df.columns:
    fig, ax = plt.subplots(figsize=(12, 3))
    if "left_pupil_diameter" in df.columns:
        ax.plot(ts, df["left_pupil_diameter"], label="left", linewidth=0.7)
    if "right_pupil_diameter" in df.columns:
        ax.plot(ts, df["right_pupil_diameter"], label="right", linewidth=0.7, alpha=0.7)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("pupil (mm)")
    ax.legend()
    plt.tight_layout()
    plt.show()
    pupil = df["left_pupil_diameter"].to_numpy(dtype=float) if "left_pupil_diameter" in df.columns else df["right_pupil_diameter"].to_numpy(dtype=float)
    sr = data.get_sample_rate() or 90.0
    res = ela.calculate_lhipa(pupil, sample_rate=sr)
    print(f"LHIPA: {res.lhipa:.4f}  valid={res.is_valid}  n={res.n_samples}")
else:
    print("No pupil columns.")

## 5. Per-phase report (SampleExperiment only)

Calibrator-only sessions show a single 'Recording' phase - that's expected.

In [ ]:
report = ela.analyze_sample_experiment(ctx.csv_path, results_json_path=ctx.results_path)
print(f"Total: {report.total_samples} samples / {report.total_duration_seconds:.2f}s @ {report.sample_rate_hz:.1f} Hz")
print(f"Phases: {list(report.phases.keys())}")
report.to_dataframe()

## 6. Post-hoc correction preview

In [ ]:
if ctx.profile_path is None:
    print("No profile JSON next to the CSV; skipping. (Notebook 08 demonstrates correction in detail.)")
else:
    profile = ela.load_profile(ctx.profile_path)
    print(f"Profile: {profile.profile_name}  schema={profile.schema_version}")
    cg = profile.combined_gaze
    print(f"  combined yaw_off={cg.gaze_yaw_offset_deg:+.3f}  pitch_off={cg.gaze_pitch_offset_deg:+.3f}")
    print(f"           gain_yaw={cg.gaze_yaw_gain:.3f}  gain_pitch={cg.gaze_pitch_gain:.3f}")